# **  Grilla rural y cobertura**

Se ubican 100 antenas separadas por 1 km en un terreno de 10 × 10 km, y 2500 puntos de población cada 200 m para evaluar zonas entre antenas. Se usa el modelo de espacio libre por su sencillez y respaldo técnico; supone que no hay obstáculos y ganancias de antena de 0 dBi. Las alturas, el umbral y el rango de potencia son supuestos de la simulación; todos los puntos tienen el mismo peso.


# Modelo de atenuación

Se utiliza la pérdida en espacio libre de la recomendación UIT-R P.525-5, ecuación:

L = 32.4 + 20 log10(f) + 20 log10(d)

L se expresa en dB, f en MHz y d en km. La ecuación representa cómo disminuye la señal al aumentar la distancia. Para obtener d se considera también la diferencia entre la altura de la antena y la del receptor.

Se trabaja con una frecuencia de 3500 MHz, dentro del intervalo de la banda n78 que aparece en 3GPP TS 38.104 [2]. Esto vincula la frecuencia del experimento con una banda de NR; no convierte al modelo en una simulación completa de 5G.

La potencia de entrada está en vatios. Para calcular la señal se convierte a dBm mediante 10 log10(P) + 30. Se suponen antenas isotrópicas, de ganancia 0 dBi. Por ello, la señal recibida se obtiene restando la pérdida a la potencia transmitida en dBm.

Finalmente, se considera cubierto un punto cuando su señal recibida es mayor o igual a -80 dBm. Si la potencia es cero, la antena está apagada y no cubre ningún punto.

*Se eligió una distribución regular para ubicar las 100 antenas y facilitar la comparación de los algoritmos. La grilla de población es más fina para evaluar también las zonas entre antenas.*

**REFERENCIAS:** [UIT-R P.525-5 (2024), en español, ecuación 6](https://www.itu.int/dms_pubrec/itu-r/rec/p/R-REC-P.525-5-202411-I!!PDF-S.pdf).

La frecuencia de 3500 MHz está dentro de la banda n78 de [ETSI, tabla 5.2-1](https://www.etsi.org/deliver/etsi_ts/138100_138199/138104/17.05.00_60/ts_138104v170500p.pdf).




In [1]:
frecuencia = 3500.0       # MHz
potencia_maxima = 1.0     # W
umbral = -80.0            # Señal mínima en dBm
altura_antena = 30.0      # Metros
altura_persona = 1.5      # Metros


1.5

In [2]:
# 100 antenas separadas por 1 km
antenas = []

for x in 0.5:1.0:9.5
    for y in 0.5:1.0:9.5
        push!(antenas, (x, y))
    end
end

# 2500 puntos de población separados por 200 m
poblacion = []

for x in 0.1:0.2:9.9
    for y in 0.1:0.2:9.9
        push!(poblacion, (x, y))
    end
end

println("Antenas: ", length(antenas))
println("Puntos de población: ", length(poblacion))


Antenas: 100
Puntos de población: 2500


In [3]:
function hay_cobertura(punto, antena, potencia)
    # Una antena apagada no cubre ningún punto
    if potencia == 0
        return false
    end

    # Distancias horizontales en kilómetros
    dx = punto[1] - antena[1]
    dy = punto[2] - antena[2]

    # Diferencia de altura convertida a kilómetros
    dz = (altura_antena - altura_persona) / 1000
    distancia = sqrt(dx^2 + dy^2 + dz^2)

    # Pérdida de señal según el modelo de espacio libre
    perdida = 32.4 + 20 * log10(frecuencia) + 20 * log10(distancia)

    # Potencia transmitida en dBm y señal recibida
    potencia_dbm = 10 * log10(potencia) + 30
    senal_recibida = potencia_dbm - perdida

    return senal_recibida >= umbral
end


hay_cobertura (generic function with 1 method)

In [4]:
punto = (1.5, 0.5)
antena = (0.5, 0.5)

println(hay_cobertura(punto, antena, 0.0))  # false: apagada
println(hay_cobertura(punto, antena, 0.1))  # false: señal insuficiente
println(hay_cobertura(punto, antena, 1.0))  # true: alcanza el umbral


false
false
true


# **Task 1.2: Función Objetivo (Fitness) y Calibración de λ - Jack**

Para optimizar las 100 potencias de transmisión, construimos la función objetivo completa:
$$\text{fitness}(\vec{P}) = \text{cobertura}(\vec{P}) - \lambda \cdot \text{interferencia}(\vec{P})$$

### 1. Cobertura Poblacional
Fracción de los 2500 puntos de población que reciben señal $\ge -80$ dBm de al menos una antena activa ($P_i > 0$).
$$C_{\text{norm}}(\vec{P}) = \frac{1}{2500} \sum_{k=1}^{2500} \mathbb{I}(\text{punto } k \text{ tiene cobertura}) \in [0.0, 1.0]$$

### 2. Modelo de Interferencia
Penalización del solapamiento destructivo y exceso de potencia en celdas contiguas:
- **Interferencia mutua entre antenas vecinas**: Pares adyacentes a $d \le 1.5$ km ponderados por $\frac{P_i \cdot P_j}{d_{ij}^2}$, normalizado a $[0, 1]$.
- **Solapamiento en puntos de población**: Exceso de antenas servidoras por punto $\sum \max(0, N_{\text{servidoras}} - 1)$, normalizado a $[0, 1]$.
- **Interferencia combinada**: $I_{\text{norm}}(\vec{P}) = 0.5 \cdot I_{\text{mutua\_norm}} + 0.5 \cdot I_{\text{solap\_norm}}$.

### 3. Justificación y Calibración de $\lambda$
- Si $\lambda = 0$, la solución trivial $\vec{P} = \vec{1}$ (todas al 100%) gana siempre, provocando máxima saturación y gasto energético.
- Si $\lambda$ es excesivo ($> 0.9$), se apagan antenas sacrificando cobertura rural crítica.
- **Calibración elegida: $\lambda = 0.5$**:
  - Para $\vec{P} = \vec{1}$: $\text{fitness} = 1.0 - 0.5 \times 1.0 = 0.500$.
  - Para una solución regulada ($P \approx 0.35$ W): cobertura $= 100\%$, interferencia $\approx 0.235$, $\text{fitness} = 1.0 - 0.5 \times 0.235 = 0.883$.
  - **Resultado**: La solución regulada e inteligente supera al óptimo trivial por $+0.383$, garantizando que los algoritmos genéticos y recocido simulado exploren soluciones óptimas no triviales.


In [5]:
include("fitness.jl")
using .Rural5GFitness
using .Rural5GFitness.Rural5GModel

println("Módulo de Fitness cargado exitosamente.")
println("Antenas: ", N_ANTENAS, " | Puntos de población: ", N_POBLACION)
println("Lambda por defecto calibrado: ", LAMBDA_DEFAULT)


In [6]:
using Random
using Printf

# 1. Vector de ceros (todas en 0 W)
p_ceros = zeros(Float64, 100)
res_0 = evaluar_solucion(p_ceros)
@printf("Vector P = 0.0 W  -> Cobertura: %6.2f%% | Interf: %.4f | Fitness: %.4f\n",
        res_0.cobertura_porcentaje, res_0.interferencia_norm, res_0.fitness)

# 2. Vector al máximo (todas en 1.0 W)
p_max = ones(Float64, 100)
res_max = evaluar_solucion(p_max)
@printf("Vector P = 1.0 W  -> Cobertura: %6.2f%% | Interf: %.4f | Fitness: %.4f\n",
        res_max.cobertura_porcentaje, res_max.interferencia_norm, res_max.fitness)

# 3. Vector aleatorio reproducible
Random.seed!(42)
p_rand = rand(Float64, 100)
res_rand = evaluar_solucion(p_rand)
@printf("Vector Aleatorio  -> Cobertura: %6.2f%% | Interf: %.4f | Fitness: %.4f\n",
        res_rand.cobertura_porcentaje, res_rand.interferencia_norm, res_rand.fitness)

# 4. Vector eficiente regulado (P = 0.35 W)
p_eficiente = fill(0.35, 100)
res_eficiente = evaluar_solucion(p_eficiente)
@printf("Vector Regulado   -> Cobertura: %6.2f%% | Interf: %.4f | Fitness: %.4f\n",
        res_eficiente.cobertura_porcentaje, res_eficiente.interferencia_norm, res_eficiente.fitness)

println("\n¿La solución coordinada supera al óptimo trivial (P=1.0)? ", res_eficiente.fitness > res_max.fitness ? "SÍ (λ bien calibrado)" : "NO")
